# 00 — Orchestrator (Paris / EUBUCCO)

Runs the full Paris land use prediction pipeline.

**Pipeline:**
```
01  Grid Definition         — EUBUCCO buildings → 150m grid + Y labels
02  Amenity Composition     — OSM amenity density + food/drink ratio
03  Building Characteristics — EUBUCCO avg height, floors, year built
04  Land Use Mix            — Shannon entropy of building subtypes
05  Tourism Intensity       — OSM tourism POI density
06  Commercial Density      — OSM shop density + brand ratio
07  ML Classification       — combine CSVs, train LR / XGBoost / RF
08  Heatmap Visualization   — static + interactive maps
```

**Data sources:**
- Y labels + building features: EUBUCCO v0.2 (European open building data)
- OSM features: Overpass API (cached locally)

**Start here:** Edit `paris.json` to change city, grid size, or target labels.

In [1]:
# ── Parameters — edit these ───────────────────────────
PARIS_CONFIG  = "paris.json"
RUN_NOTEBOOKS = ["01", "02", "03", "04", "05", "06", "07", "08"]

In [2]:
import papermill as pm
import json
import os
import time

with open(PARIS_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CSV_DIR     = config["csv_dir"]
OUTPUTS_DIR = config["outputs_dir"]
os.makedirs(CSV_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs("executed", exist_ok=True)

print(f"City:    {config['city']}")
print(f"Grid:    {config['grid_cell_size_m']}m")
print(f"Labels:  {config['target_labels']}")
print(f"CSV dir: {CSV_DIR}")
print(f"Output:  {OUTPUTS_DIR}")
print()

City:    Paris, France
Grid:    150m
Labels:  ['residential', 'commercial', 'industrial']
CSV dir: csv/Paris
Output:  outputs/Paris



In [3]:
# ── Run each notebook in sequence ─────────────────────
NOTEBOOK_MAP = {
    "01": "01_grid_definition.ipynb",
    "02": "02_amenity_composition.ipynb",
    "03": "03_building_characteristics.ipynb",
    "04": "04_land_use_mix.ipynb",
    "05": "05_tourism_intensity.ipynb",
    "06": "06_commercial_density.ipynb",
    "07": "07_ml_classification.ipynb",
    "08": "08_heatmap_visualization.ipynb",
}

results = {}
os.makedirs("executed", exist_ok=True)
for nb_id in RUN_NOTEBOOKS:
    nb_file = NOTEBOOK_MAP.get(nb_id)
    if not nb_file:
        print(f"  Unknown notebook ID: {nb_id}")
        continue
    print(f"Running {nb_file}...")
    t0 = time.time()
    try:
        pm.execute_notebook(
            nb_file,
            f"executed/{nb_file}",
            kernel_name="osmnx-scraper"
        )
        elapsed = time.time() - t0
        results[nb_id] = f"OK ({elapsed:.0f}s)"
        print(f"  {results[nb_id]}")
    except Exception as e:
        results[nb_id] = f"FAILED: {e}"
        print(f"  {results[nb_id]}")
        break

print("\n── Summary ──────────────────────────────")
for nb_id, status in results.items():
    print(f"  {NOTEBOOK_MAP[nb_id]:<40s} {status}")

d:\05- IAAC\03- Third Semester\02- Data Encoding & ML\data-encoding\OSMnx-data-scraper\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running 01_grid_definition.ipynb...


Executing: 100%|██████████| 9/9 [03:09<00:00, 21.04s/cell]


  OK (189s)
Running 02_amenity_composition.ipynb...


Executing: 100%|██████████| 8/8 [00:17<00:00,  2.24s/cell]


...

  OK (6s)
Running 07_ml_classification.ipynb...


Executing: 100%|██████████| 14/14 [00:36<00:00,  2.61s/cell]


  OK (37s)
Running 08_heatmap_visualization.ipynb...


Executing: 100%|██████████| 6/6 [12:08<00:00, 121.35s/cell]

  OK (728s)

── Summary ──────────────────────────────
  01_grid_definition.ipynb                 OK (189s)
  02_amenity_composition.ipynb             OK (18s)
  03_building_characteristics.ipynb        OK (73s)
  04_land_use_mix.ipynb                    OK (131s)
  05_tourism_intensity.ipynb               OK (7s)
  06_commercial_density.ipynb              OK (6s)
  07_ml_classification.ipynb               OK (37s)
  08_heatmap_visualization.ipynb           OK (728s)


In [4]:
# ── Merge all CSVs into combined_grid.csv ─────────────
import pandas as pd

csv_files = [
    f"{CSV_DIR}/01_grid_definition.csv",
    f"{CSV_DIR}/02_amenity_composition.csv",
    f"{CSV_DIR}/03_building_characteristics.csv",
    f"{CSV_DIR}/04_land_use_mix.csv",
    f"{CSV_DIR}/05_tourism_intensity.csv",
    f"{CSV_DIR}/06_commercial_density.csv",
]

existing = [f for f in csv_files if os.path.exists(f)]
print(f"Merging {len(existing)} CSVs...")

df = pd.read_csv(existing[0], dtype={"cell_id": str})
for f in existing[1:]:
    df_next = pd.read_csv(f, dtype={"cell_id": str})
    df = df.merge(df_next, on="cell_id", how="left")

output_path = f"{CSV_DIR}/combined_grid.csv"
df.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved combined: {output_path}  ({len(df)} rows x {df.shape[1]} cols)")
print(df.head(3).to_string())

Merging 6 CSVs...
Saved combined: csv/Paris/combined_grid.csv  (120331 rows x 16 cols)
       cell_id   cell_lat  cell_lon    zone_type  cell_building_count  amenity_density  amenity_ratio_food_drink  avg_height  avg_floors  avg_construction_year  building_count  residential_ratio  landuse_entropy  tourism_density  shop_density_km2  brand_ratio
0  r0004_c0596  48.125308  2.673187  residential                    5             0.00                       0.0         4.7         1.1                 1880.0              19              0.263           1.3838              0.0               0.0          0.0
1  r0004_c0597  48.125308  2.675239  residential                    3             0.00                       0.0         4.4         1.4                 1834.0               7              0.429           0.9852              0.0               0.0          0.0
2  r0004_c0606  48.125308  2.693706  residential                   13            88.89                       0.0         3.9         